<a href="https://colab.research.google.com/github/internshipinabook/python-internship-in-a-book/blob/main/notebooks/week7_pandas_II_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/python-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 7 -- pandas II: Cleaning, Grouping, and Transforming
### The Python Internship | Meridian Consulting

> **STARTER notebook.** Work Monday to Friday in order.
> Dataset: `golden_fork_orders.csv` (520 orders, 13 columns)

The Python Book 0 of 9 in the InternshipInABook™ Series

This repository contains the practical exercises from the book.

📖 Complete internship experience:

Workplace scenarios Mentor guidance Reflection exercises Portfolio checkpoints Capstone projects 👉 Get the complete book: https://selar.com/al990ay7ux

🚀 Start Here Welcome to The Python Internship.

If you are new to the InternshipInABook™ Series:

Run this setup notebook. Complete Week 1. Follow the weekly internship schedule. Build your portfolio. Complete the capstone project. 🔗 Repository: https://github.com/internshipinabook/python-internship-in-a-book

Run every cell top to bottom. Each cell prints ✅ if everything is working or ❌ with a fix instruction. Complete every fix before moving to Week 1.

---

<a href="https://colab.research.google.com/github/internshipinabook/python-internship-in-a-book/blob/main/notebooks/week7_pandas_II_STARTER.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>&nbsp;&nbsp;
<a href="https://github.com/InternshipInABook/book0-python/blob/main/notebooks/week7/week7_pandas_II_STARTER.ipynb">
  <img src="https://img.shields.io/badge/View%20on-GitHub-181717?logo=github" alt="View on GitHub"/>
</a>

---

## 🖥️ How to Run This Notebook

| Platform | Instructions |
|----------|-------------|
| **Google Colab** | Click the badge above — free, no setup needed |
| **Local Jupyter** | `pip install -r requirements.txt` then `jupyter lab` |
| **VS Code** | Open `.ipynb` with the Jupyter extension installed |
| **GitHub** | Click "View on GitHub" above for a read-only preview |

> **STARTER notebook** — Complete the `# YOUR CODE HERE` cells. Check the SOLUTION notebook if stuck.

In [ ]:
# ── PLATFORM SETUP ───────────────────────────────────────────────────────────
# Run this cell first. It installs missing libraries in Google Colab.
# In a local environment, skip it if requirements.txt is already installed.

import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("📦 Google Colab detected — installing libraries...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pandas>=1.5', 'numpy>=1.23', 'matplotlib>=3.6',
                    'seaborn>=0.12', 'scikit-learn>=1.2'], check=True)
    print("✅ Libraries installed.")
else:
    print("💻 Running locally.")
    print("   If you see ImportError below, run: pip install -r requirements.txt")

---

---

## This week covers

`isnull()` | `dropna()` | `fillna()` | `pd.to_datetime()` | `.astype()` | `pd.to_numeric()` | `.dt` accessor | `duplicated()` | `drop_duplicates()` | `.str` accessor | `df.rename()` | `groupby()` | `.agg()` | `reset_index()` | `.apply()` | `.map()` | `to_csv(index=False)` | cleaning log

---

## MONDAY -- Missing Values: Detect, Decide, Handle

Three questions first: What does absence mean? How much is missing? Does this column affect the analysis?

---

### Example 1 -- Detect

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Example 2 -- dropna() (use cautiously)

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Example 3 -- fillna()

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Example 4 -- Cleaning log

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


---
## MONDAY EXERCISES
---

### Exercise 7.1 -- Diagnose missing values

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Exercise 7.2 -- Handle missing values with a log

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


---
## TUESDAY -- Data Types: Converting and Fixing

---

### Example 1 -- pd.to_datetime()

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Example 2 -- astype() and category dtype

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Example 3 -- pd.to_numeric()

In [ ]:
import pandas as pd

messy = pd.Series(["1800","2200","N/A","1500","bad"])
clean = pd.to_numeric(messy, errors="coerce")
print(clean)
# "N/A" and "bad" become NaN instead of crashing

---
## TUESDAY EXERCISES
---

### Exercise 7.3 -- Convert the date column

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Exercise 7.4 -- Type conversions

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


---
## WEDNESDAY -- Duplicates and String Cleaning

---

### Example 1 -- Duplicates

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Example 2 -- .str accessor

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Example 3 -- df.rename() with a copy

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


---
## WEDNESDAY EXERCISES
---

### Exercise 7.5 -- Check and clean

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Exercise 7.6 -- String operations

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


---
## THURSDAY -- Groupby and Aggregation

`groupby()` = split → apply → combine.

---

### Example 1 -- Basic groupby

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Example 2 -- .agg()

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Example 3 -- Daily summary

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


---
## THURSDAY EXERCISES
---

### Exercise 7.7 -- Basic groupby

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Exercise 7.8 -- agg() and multi-level

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


---
## FRIDAY -- Adding Columns and the Cleaned Output

---

### Example 1 -- apply() and map()

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### The Friday Build -- Complete Cleaning Pipeline

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/supermarket_sales.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/python-internship-in-a-book/main/data/supermarket_sales.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload supermarket_sales.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        df = pd.DataFrame({
            'Invoice ID':    [f'INV-{i:04d}' for i in range(100)],
            'Branch':        list('ABCAABCBCA' * 10),
            'City':          ['Yangon','Mandalay','Naypyitaw','Yangon','Yangon',
                              'Mandalay','Naypyitaw','Mandalay','Naypyitaw','Yangon'] * 10,
            'Customer type': ['Member','Normal'] * 50,
            'Gender':        ['Male','Female'] * 50,
            'Product line':  ['Health and beauty','Electronic accessories',
                              'Food and beverages','Sports and travel',
                              'Home and lifestyle','Fashion accessories'] * 17 + ['Food and beverages','Food and beverages'],
            'Unit price':    list(np.round(np.random.uniform(10,100,100),2)),
            'Quantity':      list(np.random.randint(1,11,100)),
            'Total':         list(np.round(np.random.uniform(50,500,100),2)),
            'Rating':        list(np.round(np.random.uniform(4,10,100),1)),
        })
        print('Sample data ready (100 rows) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


### Extension challenges

In [ ]:
# Challenge A: Server performance table
# groupby server: total revenue, order count, avg order
# Save as server_performance.csv

# Challenge B: pd.cut() for revenue_tier
# bins   = [0, 1000, 2500, 5000, 8000, float("inf")]
# labels = ["Standard","Bronze","Silver","Gold","Platinum"]
# Does pd.cut() give same result as apply(get_tier)?

# Challenge C: transform() for weekly contribution
# df["weekly_rev"] = df.groupby("week")["total_price"].transform("sum")
# df["pct_of_week"] = df["total_price"] / df["weekly_rev"] * 100
# Which order is highest % of its week?


## ✅ What You Can Do After This Week

By the end of this week, you can:
- Merge two DataFrames on a shared key column — the equivalent of a SQL JOIN
- Reshape data between long and wide format using `melt()` and `pivot_table()`
- Sort, rank, and reindex data for presentation-ready output
- Apply custom transformation functions to a DataFrame with `.apply()`
- Produce a cross-tabulation summary that answers multi-dimensional business questions

*Add `week7_pandas_II.ipynb` to your internship portfolio.*

---

## Week 7 Checklist

- [ ] Mon 7.1-7.2: 105 missing found; fillna(0) applied; logged; saved v1
- [ ] Tue 7.3-7.4: date to datetime; month/week extracted; table_number to int; saved v2
- [ ] Wed 7.5-7.6: 0 duplicates confirmed; whitespace stripped; title-case applied
- [ ] Thu 7.7-7.8: groupby by server, time, item; agg() multi-stat; daily summary saved
- [ ] Fri: pipeline runs; clean_final.csv produced; cleaning_log.txt written
- [ ] One extension done
- [ ] Can explain: dropna vs fillna, index=False, groupby split-apply-combine, apply vs map, cleaning log purpose

---
## Week 7 Complete
Next: `week8/week8_*_STARTER.ipynb`

---
*The Python Internship | Meridian Consulting*